# 03_consolidate_embeddings

**Objective:** Assemble one embedding per Lens ID, pulling vectors for mapped patents from the EPO/USPTO PaECTER parquets and for uncovered patents from the in-house recovered embeddings.

**Inputs:** EPO/USPTO PaECTER embedding parquets; `lens_to_familyid.parquet`; `recovered_embeddings.parquet`; `startup_patent_matches.csv`.

**Outputs:** `lens_id_to_embedding.parquet` (`lens_id`, `embedding`, `source`).

In [1]:
# Imports
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [2]:
# Paths
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists())
RAW  = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)
EPO       = RAW / "embeddings" / "EPO_PaECTER_embeddings.parquet"
USP       = RAW / "embeddings" / "USPTO_PaECTER_embeddings.parquet"
MAPPING   = PROC / "lens_to_familyid.parquet"
RECOVERED = PROC / "recovered_embeddings.parquet"
MATCHES   = PROC / "startup_patent_matches.csv"
OUTPUT    = PROC / "lens_id_to_embedding.parquet"

## Determine which lens_ids are needed

In [3]:
# Determine which lens_ids are needed (mapped, in-sample, with a matched publno)
print("=== Lade Mappings ===")
matches = pd.read_csv(MATCHES, usecols=["lens_id"])
needed = set(matches["lens_id"].dropna().unique())
print(f"  unique lens_ids in matches: {len(needed):,}")

mapping = pd.read_parquet(MAPPING, engine="fastparquet")
mapping = mapping[mapping["lens_id"].isin(needed)
                  & mapping["docdb_family_id"].notna()
                  & mapping["matched_publno"].notna()].copy()
print(f"  davon gemappt (mit publno) : {len(mapping):,}")

need_epo = set(mapping.loc[mapping["source"]=="EPO",   "matched_publno"])
need_usp = set(mapping.loc[mapping["source"]=="USPTO", "matched_publno"])
print(f"  benoetigte EPO-publnos     : {len(need_epo):,}")
print(f"  benoetigte USPTO-publnos   : {len(need_usp):,}")

=== Lade Mappings ===
  unique lens_ids in matches: 42,503
  davon gemappt (mit publno) : 31,361
  benoetigte EPO-publnos     : 24,964
  benoetigte USPTO-publnos   : 2,778


## Stream embeddings from PaECTER parquets

In [4]:
# Collect embeddings for the wanted publnos from a PaECTER parquet
def collect_embs(path, want):
    print(f"\n  Scan {path.name} ({len(want):,} publnos gesucht) ...")
    out = {}
    pf = pq.ParquetFile(path)
    for batch in pf.iter_batches(batch_size=500_000,
                                 columns=["embedding","publno"]):
        df = batch.to_pandas()
        sel = df["publno"].isin(want)
        if sel.any():
            for _, r in df[sel].iterrows():
                if r["publno"] not in out:
                    out[r["publno"]] = np.asarray(r["embedding"], dtype=np.float32)
        if len(out) >= len(want):
            break
    print(f"    gefunden: {len(out):,} / {len(want):,}")
    return out

print("\n=== Embeddings aus PaECTER-Parquets holen ===")
epo_emb = collect_embs(EPO, need_epo)
usp_emb = collect_embs(USP, need_usp)


=== Embeddings aus PaECTER-Parquets holen ===

  Scan EPO_PaECTER_embeddings.parquet (24,964 publnos gesucht) ...
    gefunden: 24,964 / 24,964

  Scan USPTO_PaECTER_embeddings.parquet (2,778 publnos gesucht) ...


OSError: [Errno 89] Error reading bytes from file. Detail: [errno 89] Operation canceled

## Resolve to lens_id level

In [ ]:
# Resolve each mapped lens_id to its embedding
print("\n=== Auf lens_id-Ebene auswerten ===")
records = []
miss_lookup = 0
for _, r in mapping.iterrows():
    src, pub = r["source"], r["matched_publno"]
    vec = (epo_emb if src=="EPO" else usp_emb).get(pub)
    if vec is None:
        miss_lookup += 1
        continue
    records.append((r["lens_id"], vec, src))

print(f"  PaECTER-resolved  : {len(records):,}")
print(f"  Lookup-Verlust    : {miss_lookup}")

## Add recovered embeddings

In [ ]:
# Append the in-house recovered embeddings
rec = pd.read_parquet(RECOVERED)
rec = rec[rec["lens_id"].isin(needed)].copy()
print(f"  Recovered (Schritt 3b): {len(rec):,}")

for _, r in rec.iterrows():
    records.append((r["lens_id"], np.asarray(r["embedding"], dtype=np.float32), "RECOVERED"))

## Dedup and save

In [ ]:
# Deduplicate lens_ids and write the consolidated parquet
print("\n=== Dedup + Speichern ===")
df = pd.DataFrame(records, columns=["lens_id","embedding","source"])
n_before = len(df)
df = df.drop_duplicates(subset="lens_id", keep="first")
print(f"  Zeilen vor dedup : {n_before:,}")
print(f"  Zeilen nach dedup: {len(df):,}")

table = pa.table({
    "lens_id":   pa.array(df["lens_id"].tolist(),   type=pa.string()),
    "embedding": pa.array(df["embedding"].tolist(), type=pa.list_(pa.float32())),
    "source":    pa.array(df["source"].tolist(),    type=pa.string()),
})
pq.write_table(table, OUTPUT, compression="zstd")
size_mb = OUTPUT.stat().st_size / 1e6
print(f"\n  -> {OUTPUT}  ({size_mb:.1f} MB)")

## Final coverage report

In [ ]:
# Report final coverage
print("\n=== Final Coverage ===")
print(df["source"].value_counts().to_string())
print(f"\n  total covered lens_ids : {len(df):,} / {len(needed):,}  "
      f"({len(df)/len(needed)*100:.2f}%)")
print(f"  uncovered              : {len(needed) - len(df):,}")